# 07 — Consultation de la récupération constructive

Notebook de lecture des traces privées, configurable par `CGR_CONSTRUCTIVE_ROOT`.
**Exécuter tout ne lance ni simulation ni entraînement.** Les figures sont
reconstruites à partir des positions natives. Les tableaux et audits sont lus dans
le dossier privé ; ce modèle public ne contient aucun résultat embarqué.

Une récupération physique doit être distinguée de la conformité aux contraintes,
et un pilote technique d'une validation indépendante de généralisation.

In [ ]:
from pathlib import Path
import os, json, html
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from cav_recovery.recovery.inspection import ConstructiveStudy

ROOT = Path(os.environ.get('CGR_CONSTRUCTIVE_ROOT',
    'D:/cav-recovery-data/recovery-constructive-20261021'))
study = ConstructiveStudy(ROOT)
FIGURES = ROOT / 'evidence' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
def read(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))
def table(records, columns):
    head = ''.join('<th>' + html.escape(c) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(r.get(c, ''))) +
        '</td>' for c in columns) + '</tr>' for r in records)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))
def show(figure, name):
    figure.savefig(FIGURES / (name + '.png'), dpi=145)
    figure.savefig(FIGURES / (name + '.svg'))
    display(figure)
    plt.close(figure)
print('Sources privées :', ROOT)
print('Figures régénérables :', FIGURES)

## Diagnostic des anciens essais à 20 CAV

Des reculs ne suffisent pas si le véhicule ne libère pas l'espace requis ou ne peut
reprendre sa mission. Le diagnostic distingue le frein maintenu au repos numérique,
les maintiens de voisins trop éloignés, les contraintes géométriques et les gains partiels.
Les corps et seuils M02/K04 historiques ne sont pas modifiés.

In [ ]:
old = read('diagnostic-initial.json')
table([dict(trial=r['trial'], **r['cooperative_counts']) for r in old],
      ['trial', 'cooperative_hold', 'hold_for_intent_beyond_required_ack_radius'])
table([dict(trial=r['trial'], **r['counts']) for r in read('old-control-counts.json')],
      ['trial', 'positive_accepted_commands', 'positive_command_braked_at_negative_numerical_rest'])

## Tous les essais conservés

L'inventaire inclut les explorations infructueuses et la collision d'un essai intermédiaire.
Une arrivée ou une fraction libérée ne suffit pas à établir une récupération durable.
Les variantes de frein maximal rejetées et le premier pilote v1 restent dans le dossier privé.

In [ ]:
table(study.inventory(), ['run', 'method', 'introduced', 'arrivals', 'errors',
    'reverse_max_m', 'global_actor', 'cpu_s'])

## Comparaisons locales appariées

REC-EVAL-1 mesure la progression nette d'une longueur pendant 10 s puis le suivi
sans rechute pendant 30 s, ou les arrivées terminales vérifiées. La population A0
est fixée par la référence nominale, hors acteur. Le contrôle d'appariement vérifie
le préfixe physique avant la première intervention. L'activation expérimentale
locale ne prétend pas être une confirmation autonome K04.

In [ ]:
proofs = []
for filename, name in [('evidence/local-four-strict-pairing.json', '4 CAV'),
                       ('evidence/local-eight-strict-pairing.json', '8 CAV')]:
    r = read(filename)
    proofs.append(dict(scene=name, A0=r['intervention']['affected'], t_phys=r['t_phys_s'],
        t_action=r['t_act_s'], nominal=r['nominal']['status'], local=r['intervention']['status'],
        t_confirmation=r['intervention']['t_conf_s'],
        t_validation=r['intervention'].get('t_final_validation_s'),
        prefixe_identique=r['identical_physical_reference_until_action']))
table(proofs, ['scene','A0','t_phys','t_action','nominal','local','t_confirmation',
               't_validation','prefixe_identique'])
show(study.plot_comparison('evidence/local-four-strict-pairing.json'), '01_recuperation_locale_4')
show(study.plot_comparison('evidence/local-eight-strict-pairing.json'), '02_gain_partiel_8')
display(read('evidence/physical-repeatability.json'))

## Voir les séquences physiques

Les silhouettes respectent 4,515 × 1,852 m. Les vues fixes cadrent la zone centrale ;
le lecteur HTML couvre aussi les approches. Il permet pause, vitesse et retour temporel.
Les références globales utilisées pour dessiner et évaluer ne sont pas des entrées acteur.

In [ ]:
show(study.plot_sequence('progression-oblique-4-v3/nominal'), '03_sequence_nominale')
show(study.plot_sequence('progression-oblique-4-v3/cooperative'), '04_sequence_locale')
show(study.plot_movement('progression-oblique-4-v3/cooperative'), '05_mouvements_reels')
player = study.write_player([
    'progression-oblique-4-v3/nominal', 'progression-oblique-4-v3/cooperative',
    'progression-oblique-8-v3/cooperative',
    'graph-pilot-v2/episodes/episode-000', 'graph-pilot-v2/episodes/episode-001'],
    ROOT / 'evidence' / 'LECTEUR_SEQUENCES.html')
print('Ouvrir dans le navigateur :', player)

## Dépendances, demandes d'espace et engagements

Le graphe ci-dessous est une **annotation physique globale hors ligne**, distincte
du C-RDG local partiel reçu par l'acteur. CLEAR_REQUEST est une demande déclarée,
pas une relation causale confirmée. Les maintiens et accords opérationnels sont limités
au voisinage de 10 m ; le transport V2V reste à 30 m.

In [ ]:
show(study.plot_dependencies('progression-oblique-4-v3/cooperative'), '06_dependances_physiques')
show(study.plot_protocol('progression-oblique-4-v3/cooperative'), '07_protocole_v2v')
table([dict(run=k, **v) for k,v in read('evidence/commands-audit.json').items()],
      ['run','total','accepted','refusals'])

## Délais d'activation et faisabilité globale séparée

Une intervention avant confirmation de persistance ne doit pas être comptée comme
récupération d'une obstruction persistante. K04 reste inchangé.
Une recherche globale doit déclarer ses privilèges et les éventuelles différences
antérieures à la manœuvre. Le comparateur strict refuse les préfixes divergents.

In [ ]:
timings = []
for delay in [5,15,35]:
    r = read(f'timing-{delay}-comparison.json')
    timings.append(dict(delai_local_s=delay, action_s=r['t_act_s'],
        effet_avant_action=r['physical_effects_present_before_action'],
        incident_persistant_avant_action=r['qualified_incident_present_before_action'],
        resultat=r['intervention']['status']))
table(timings, ['delai_local_s','action_s','effet_avant_action',
                'incident_persistant_avant_action','resultat'])
g = read('evidence/global-feasibility-only.json')
display({k:v for k,v in g.items() if k!='result'})
display({k:v for k,v in g['result'].items() if k!='timeline'})

## Pilote hiérarchique Graph-PPO

Encodeur partagé à deux passages, acteur et critique locaux, sans GRU. Le réseau
choisit les options et les réponses accord/refus ; l'armement et la négociation
restent une couche finie commune. Les acteurs voient leur état, mission, perception,
commandes et informations locales valides, sans A0 global ni annotation future.
La récompense partagée vient d'une instrumentation après action, hors entrées acteur.

Les retours de très petits lots conservent un gradient de politique même lorsqu'une
seule décision a été collectée. La reprise est vérifiée à une frontière d'épisode,
avec sauvegarde des poids, de l'optimiseur et des états aléatoires.
Le pilote porte sur le développement connu, sans utilisation du test réservé.

In [ ]:
audit = read('evidence/pilot-audit.json')
table(audit['episodes'], ['episode','decisions','agents','phase_counts','physical_outcome',
      'all_arrivals_s','reward','reverse_budget_excess_m','protocol_valid'])
display(audit['resume'])
display(read('graph-pilot-v2/complete.json'))
show(study.plot_pilot(), '08_choix_et_recompenses')
print('La reprise est à une frontière d’épisode ; aucun état Bullet intermédiaire n’est restauré.')

## Intégrité et limites

Consulter l'audit de budget en complément de l'absence de collision : un résultat
physique durable n'établit pas seul la conformité de toutes les contraintes.
Le passage à une complexité supérieure exige une validation locale préalable.
Les manifestes contrôlent les fichiers déclarés et snapshots ; le contrôle des
références historiques ne doit pas être confondu avec un rehash intégral de M02.

In [ ]:
display(read('evidence/trace-integrity.json'))
p = read('evidence/preservation-after.json')
display({k:v for k,v in p.items() if k not in ['current_refs','current_worktrees','intentional_changes']})
raw_log = (ROOT/'tests-final.log').read_bytes()
print(raw_log.decode('utf-16' if raw_log.startswith(bytes([255,254])) else 'utf-8').splitlines()[-1])
print('Rapports et provenance : INDEX_IMPLEMENTATION.md, RAPPORT_FINAL.md, REVUE_METHODES.md')

## Démonstration et reproduction volontaire

1. Ouvrir le lecteur `evidence/LECTEUR_SEQUENCES.html` ou ce notebook exécuté privé.
2. Consulter les JSON de `evidence/` pour les métriques et les anomalies.
3. Pour reconstruire une spécification sans simuler : appeler
   `specification('collective-3', seed=21021, horizon_s=180.)`, garder les 4 premiers
   `agents`, fixer `active_limit=4` et `phase='recovery_constructive_development'`.
   La recette complète est dans `configs/recovery/constructive-study-v2.json`.
4. Un rejeu explicitement demandé utilise `cgr recovery run spec.json --method nominal`
   ou `--method cooperative --output D:/.../nouvel-essai`. Ne jamais écraser les traces.
5. Ne pas lancer ou poursuivre un pilote sans lire ses audits privés et valider ses protections.
   La commande documentée est `cgr recovery pilot --config configs/recovery/pilot-graph-v2.json`
   avec `--proof` vers la preuve locale et `--output` vers un dossier privé neuf.
   `--stop-after 1` provoque le code 86 intentionnel ; la même commande sans cette option reprend.

Lire `REVUE_METHODES.md` pour les sources primaires Qi et Goto et leurs différences
d'hypothèses. Les observations actuelles ne valent ni validation indépendante ni résultat dense.